In [0]:
%run ../common/01_config

In [0]:
%run ../common/02_adls_connection

In [0]:
batch_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(bronze_batch_online_retail)
)

display(batch_df)

In [0]:
batch_df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim, upper, to_timestamp, current_timestamp

batch_silver_df = (
    batch_df
    .dropDuplicates()
    .filter(col("InvoiceNo").isNotNull())
    .filter(col("StockCode").isNotNull())
    .filter(col("Quantity").isNotNull())
    .filter(col("UnitPrice").isNotNull())
    .withColumn("InvoiceNo", trim(col("InvoiceNo")))
    .withColumn("StockCode", trim(col("StockCode")))
    .withColumn("Description", upper(trim(col("Description"))))
    .withColumn("Country", upper(trim(col("Country"))))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("UnitPrice", col("UnitPrice").cast("double"))
    .withColumn("CustomerID", col("CustomerID").cast("long"))
    .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate")))
    .withColumn("TotalAmount", col("Quantity") * col("UnitPrice"))
    .withColumn("processed_at", current_timestamp())
)

In [0]:
batch_silver_df = (
    batch_silver_df
    .filter(col("Quantity") > 0)
    .filter(col("UnitPrice") > 0)
)

display(batch_silver_df)

In [0]:
batch_silver_df.write \
    .format("delta").mode("overwrite").save(silver_batch_online_retail)

In [0]:
silver_batch_df = spark.read.format("delta").load(silver_batch_online_retail)

print("Silver batch count:", silver_batch_df.count())

display(silver_batch_df)